# Import 

In [ ]:
# EXCERCISE 1:
# Import data:
import string
import re
import pandas as pd
import numpy as np
data_1 = pd.read_csv(r'/Users/ngocta/Desktop/CoderSchool-AI/Week5/BTVN_1/dataset_1.csv', sep = ',')

#Preprocessing data
clean_data_1 = data_1.copy()
num_cols = ['Population', 'Area KM2', 'Area   M2','Density KM2','Density  M2']

# Check for text columns:
for col in clean_data_1.columns:
    if clean_data_1[col].dtype == object:
        for punc in string.punctuation:
            # Remove special characters 
            clean_data_1[col] = clean_data_1[col].astype(str).str.replace(punc, "", regex = False)
            # Remove leading/trailing white space:
            clean_data_1[col] = clean_data_1[col].str.strip()

# Convert number columns to correct format and fill non-numeric values to NaN:
num_check = clean_data_1[num_cols].apply(lambda x: pd.to_numeric(x, errors = 'coerce'))

# Check which rows containing NaN
rows_with_Nan = num_check.isna().any(axis =1)

# Remove rows containing NaN:
clean_data_1 = clean_data_1[~rows_with_Nan]

# Convert number columns to the correct format:
clean_data_1[num_cols] = clean_data_1[num_cols].apply(pd.to_numeric, errors = 'raise')

print(clean_data_1)
print(clean_data_1.dtypes)

In [ ]:
#Task 1: Sắp xếp và in ra 10 thành phố có dân số lớn nhất & 10 thành phố có dân số nhỏ nhất
top_10 = clean_data_1.sort_values(by='Population', ascending=False).head(10).reset_index(drop = True)
bottom_10 = clean_data_1.sort_values(by = 'Population', ascending = True).head(10).reset_index(drop = True)

print("Top 10 thành phố có dân số lớn nhất:")
print(top_10)

print("\nTop 10 thành phố có dân số nhỏ nhất:")
print(bottom_10)

In [ ]:
#Task 2: In ra tên các quốc gia có tối thiểu 3 thành phố trong danh sách này
result_Ex1_2 = clean_data_1.groupby('Country').agg({'City': lambda x : x.nunique()}).reset_index()
result_Ex1_2 = result_Ex1_2[result_Ex1_2['City'] >=3]['Country'].tolist()
print(result_Ex1_2)

In [ ]:
#Task 3: In ra Top 5 quốc gia có nhiều thành phố xuất hiện trong bảng này nhất
result_Ex1_3 = clean_data_1.groupby('Country').agg({'City': lambda x:x.nunique()}).sort_values(by = 'City', ascending = False).head(5)
print(result_Ex1_3)

In [ ]:
#Task 4: In ra các thành phố có dân số & diện tích đều nằm trong Top 20
top_20_pop = clean_data_1.sort_values(by = 'Population', ascending = False).head(20).reset_index(drop = True)
top_20_area= clean_data_1.sort_values(by = 'Area KM2', ascending = False).head(20).reset_index(drop = True)
# Find the common cities between 2 lists:
top_city_list = set(top_20_pop['City']) & set(top_20_area['City'])
result_Ex1_4 = clean_data_1[clean_data_1['City'].isin(top_city_list)].reset_index(drop = True)

# Extract relevant columns:
result_Ex1_4 = result_Ex1_4[['City', 'Population', 'Area KM2']]
print(result_Ex1_4)

In [ ]:
#Task 5: Thống kê mật độ dân số theo quốc gia
result_Ex1_5 = clean_data_1.groupby('Country')['Density KM2'].mean().round().to_frame(name = 'Avg Density KM2')
print(result_Ex1_5)

In [ ]:
#Task 6: Thống qua các thành phố có dân số lớn nhất của từng quốc gia (chỉ tính của những quốc gia có 2
#thành phố xuất hiện trở lên trong bảng)
city_count = clean_data_1.groupby('Country')['City'].nunique()

# Filter country with 2 or more cities:
country_list = city_count[city_count >=2].index
filtered_countries = clean_data_1[clean_data_1['Country'].isin(country_list)]

# Get the city with the most population for each country:
result_Ex1_6 = filtered_countries.loc[filtered_countries.groupby('Country')['Population'].idxmax()]

# Extract the relevant columns:
result_Ex1_6 = result_Ex1_6[['Country','City','Population']].sort_values(by = 'Population', ascending = False).reset_index(drop = True)
print(result_Ex1_6)

In [ ]:
# EXCERCISE 2:
# Import data:
data_2 = pd.read_csv(r'/Users/ngocta/Desktop/CoderSchool-AI/Week5/BTVN_2/dataset_2.csv', sep = ',')
print(data_2)
print(data_2.dtypes)

In [ ]:
# Task 1: In ra index của 10 học sinh có math score và reading score đều nằm trong Top 100
top_100_math = data_2.sort_values(by = 'math score', ascending = False).head(100)
top_100_reading = data_2.sort_values(by = 'reading score', ascending = False).head(100)
top_10 = data_2.loc[data_2.index.isin(set(top_100_math.index) & set(top_100_reading.index))].head(10)
print(top_10)

In [ ]:
# Task 2: In ra index của 20 học sinh có điểm writing score cao nhất của mỗi group (race/ethnicity)
top_20_writing = data_2.sort_values(by = 'writing score', ascending = False).groupby('race/ethnicity').head(20)

print(top_20_writing.index)

In [ ]:
# Task 3: Thống kê số lượng học sinh của group A theo parental level ofeducation
result_Ex2_3 = data_2[data_2['race/ethnicity'] == 'group A']['parental level of education'].value_counts()
print(result_Ex2_3)

In [ ]:
# Task 4: Có phải xu hướng chung, cha mẹ có học vấn càng cao thì điểm số trung bình 3 môn 
# của con cái cũng càng cao ko

# Find the average of the 3 scores:
data_2['avg_score'] = data_2[['math score', 'reading score', 'writing score']].mean(axis=1).round(1)

# Find the average for each parental education level:
result_Ex2_4 = data_2.groupby('parental level of education')['avg_score'].mean().round(1)
print(result_Ex2_4)

# Kết luận: khi so sánh giữa các bận học vấn thì cha mẹ có bậc học vấn càng cao thì điểm số 
# trung bình 3 môn của con cái càng cao

In [ ]:
# Task 5: Chất lượng bữa ăn (lunch) có tương phản với điểm số trung bình 3 môn của học sinh không
result_Ex2_5 = data_2.groupby('lunch')['avg_score'].mean().round(1)
print(result_Ex2_5)

#Kết luận: học sinh có bữa ăn tốt có xu hướng có điểm số tốt hơn 

In [ ]:
# Task 6: Tìm ra Top 10 học sinh có điểm toán cao nhất của mỗi group
top_math_by_group = data_2.groupby('race/ethnicity').apply(lambda x: x[['race/ethnicity','math score']].sort_values(by = 'math score', ascending = False).head(10))
print(top_math_by_group)

In [ ]:
# Task 7: Với những học sinh ở cùng 1 group, cùng parental level ofeducation, cùng loại bữa ăn (lunch),
# tham gia test preparation course có giúp học sinh đó có điểm trung bình 3 môn cao hơn những 
# học sinh không tham gia ko
result_Ex2_7 = data_2.groupby(['race/ethnicity', 'parental level of education', 'lunch','test preparation course'])['avg_score'].mean().reset_index()
print(result_Ex2_7.head(50))

# Kết luận: đối với những những học sinh cùng 1 group, cùng parental level ofeducation, cùng loại bữa ăn (lunch),
# tham gia test preparation course có xu hướng giúp học sinh đó đạt điểm cao hơn

In [ ]:
# EXERCISE 3:
data_3 = pd.read_csv(f'/Users/ngocta/Desktop/CoderSchool-AI/Week5/BTVN_3/dataset_3.csv', sep = ',')
print(data_3)
print(data_3.dtypes)

In [ ]:
# Task 1: Liệt kê các sản phẩm có doanh thu lớn hơn 100.000 Euro
data_3['revenue'] = data_3['retail_price']*data_3['units_sold']
result_Ex3_1 = data_3[data_3['revenue'] > 100000]
print(result_Ex3_1)

In [ ]:
# Task 2: Liệt kê các sản phẩm có units_sold nằm trong top 100, đồng thời nằm trong
# nhóm 10% sản phẩm có rating cao nhất

top_10_unit_sold = data_3.sort_values(by = 'units_sold', ascending = False).head(100)
top_10pct_rating = data_3[data_3['rating'] > np.percentile(data_3['rating'], 90)]

# Find the products in both lists:
result_Ex3_2 = data_3[data_3.index.isin(set(top_10_unit_sold.index) & set(top_10pct_rating.index))]
print(result_Ex3_2)

In [ ]:
# Task 3: Có phải sản phẩm càng được rating nhiều thì càng bán được nhiều sản phẩm ko
result_Ex3_3 = data_3['units_sold'].corr(data_3['rating_count'])
print(result_Ex3_3)

# Kết luận: có sự tương quan cao giữa số sản phẩm bán được và số lượng review (càng nhiều rating, càng bán được nhiều)

In [ ]:
# Task 4: Thống kê Top 50 sản phẩm nằm trong rating từ 3.0 → 4.0 bán chạy nhất
result_Ex3_4 = data_3[(data_3['rating']>=3) & (data_3['rating']<=4)].sort_values(by = 'units_sold', ascending = False).head(50)
print(result_Ex3_4)

In [ ]:
# Task 5: Có bao nhiêu sản phẩm có giá bán lớn hơn 50 EURO
result_Ex3_5 = len(data_3[data_3['retail_price']>50])
print(f"Số sản phẩm có giá bán >50 Euro: {result_Ex3_5} sản phẩm")

In [ ]:
# EXERCISE 4:
# Import data:
data_4 = pd.read_csv(f'/Users/ngocta/Desktop/CoderSchool-AI/Week5/ds_salaries.csv', sep = ',')

# Clean data:
for col in data_4.columns:
    if data_4[col].dtype == 'object':
            data_4[col] = data_4[col].str.replace('ML', 'Machine Learning', case = True).str.strip()

print(data_4)
print(data_4.dtypes)

In [ ]:
# Task 1: Job title nào có mức lương trung bình theo USD cao nhất
result_Ex4_1 = data_4.groupby('job_title')['salary_in_usd'].mean().round()
result_Ex4_1 = result_Ex4_1[result_Ex4_1 == result_Ex4_1.max()]
print(result_Ex4_1)

In [ ]:
# Task 2: Data Scientist mang quốc tịch nào có mức lương trung bình cao nhất
data_scientist_list = data_4[data_4['job_title'] == 'Data Scientist']
ds_highest_salary = data_scientist_list.groupby('company_location')['salary_in_usd'].mean().round()
result_Ex4_2 = ds_highest_salary[ds_highest_salary == ds_highest_salary.max()]

print(f'Data Scientist có mức lương trung bình cao nhất ở {result_Ex4_2.index[0]} là {result_Ex4_2.values[0]} USD')

In [ ]:
# Task 3:Có phải ở USA, nhân sự ngoại quốc nhận được mức lương trung bình cao hơn nhân sự trong nước không
company_in_usa = data_4[data_4['company_location'] == 'US']
company_in_usa['is_us_staff'] = company_in_usa['employee_residence'].apply(lambda x : 'Yes' if x == 'US' else 'No')
result_Ex4_3 = company_in_usa.groupby('is_us_staff')['salary_in_usd'].mean().round()
print(result_Ex4_3)

# Kết luận: ở USA, nhân sự ngoại quốc có mức lương trung bình thấp hơn nhân sự trong nước

In [ ]:
# Task 4: Mức lương trung bình của Data Engineer (SE level) ở các công ty nhỏ và vừa ở USA
DA_list = data_4[(data_4['company_location'] == 'US') & (data_4['company_size'].isin(['S','M'])) & (data_4['job_title'] == 'Data Engineer') & (data_4['experience_level'] == 'SE')]
result_Ex4_4 = DA_list['salary_in_usd'].mean().round()

print(f'Mức lương trung bình của Data Engineer (SE level) ở công ty nhỏ và vừa ở USA: {result_Ex4_4} USD')

In [ ]:
# Task 5: So sánh mức lương của Data Scientist ở USA ở các trình độ (experience_level) khác nhau
DA_USA = data_4[(data_4['company_location'] == 'US') & (data_4['job_title'] == 'Data Scientist')]
result_Ex4_5 = DA_USA.groupby('experience_level')['salary_in_usd'].mean().round()
print(result_Ex4_5)

In [ ]:
# Task 6: Nhân sự là người nước ngoài chiếm tỉ lệ bao nhiêu phần trăm ở các công ty lớn ở USA
L_company_USA = company_in_usa[company_in_usa['company_size'] == 'L']
staff_count = L_company_USA['is_us_staff'].value_counts()
result_Ex4_6 = round((staff_count.get('No', 0) / staff_count.sum())*100, 1)
print(f'Tỷ lệ nhân sự là người nước ngoài  ở các công ty lớn ở USA: {result_Ex4_6}%')

In [ ]:
# Task 7: Tìm ra 3 job title có mức lương trung bình cao nhất ở level MI cho nhân sự trong nước
job_list = company_in_usa[company_in_usa['experience_level'] == 'MI']
result_Ex4_7 = job_list.groupby('job_title')['salary_in_usd'].mean().round().sort_values(ascending = False).head(3)
print(result_Ex4_7)

In [ ]:
# Task 8: Job nào ở Germany (DE) có mức lương trung bình thấp nhất
company_in_DE = data_4[data_4['company_location'] == "DE"]
salary_in_DA = company_in_DE.groupby('job_title')['salary_in_usd'].mean().round()
result_Ex4_8 = salary_in_DA[salary_in_DA == salary_in_DA.min()]
print(result_Ex4_8)

In [ ]:
# Task 9: Job nào có khoảng cách thu nhập giữa SE và MI ít nhất
# Get all jobs with SE & MI level:
SE_MI_job_list = data_4[data_4['experience_level'].isin(['SE','MI'])]
SE_MI_job_list = SE_MI_job_list.groupby(['job_title', 'experience_level'])['salary_in_usd'].mean().round().reset_index()

# Filter jobs with both SE & MI levels:
valid_SE_MI_jobs = SE_MI_job_list['job_title'].value_counts().loc[lambda x: x==2].index
final_SE_MI_list = SE_MI_job_list[SE_MI_job_list['job_title'].isin(valid_SE_MI_jobs)]

# Pivot table & find the salary gap between MI & SE for each job title:
pivoted_list = final_SE_MI_list.pivot(index = 'job_title', columns = 'experience_level', values = 'salary_in_usd')
pivoted_list['salary_gap'] = abs(pivoted_list['MI'] - pivoted_list['SE'])
pivoted_list = pivoted_list.sort_values(by = 'salary_gap')
result_Ex4_9 = pivoted_list[pivoted_list['salary_gap'] == pivoted_list['salary_gap'].min()].reset_index()

print(f"Job có khoảng cách thu nhập giữa SE và MI ít nhất:")
for index, row in result_Ex4_9.iterrows():
    print(f"{row['job_title']} với gap thu nhập {row['salary_gap']} USD")

In [ ]:
# Task 10: Job nào có mức lương trung bình đều nằm trong top 5 ở tất cả các level
#Get the top 5 job of each experience level
salary_by_exp = data_4.groupby('experience_level').apply(lambda x: x.groupby('job_title')['salary_in_usd'].mean().round().sort_values(ascending = False).head(5)).reset_index()

# Count the occurrances of each top job
job_count = salary_by_exp['job_title'].value_counts()

# Count the number of experience level
exp_level_count = data_4['experience_level'].nunique()

# Check if there's any job in the top 5 list of all experience level:
result_Ex4_10 = [job for job, count in job_count.items() if count == exp_level_count]
if not result_Ex4_10:
    print("Không có job nào lương trung bình đều nằm trong top 5 ở tất cả các level")
else:
    print("Job có lương trung bình đều nằm trong top 5 ở tất cả các level:")
    for job in result_Ex4_10:
        print(job)

In [ ]:
# Task 11: Job nào có mức chênh lệch thu nhập trung bình giữa EN & EX level lớn nhất (chỉ tính các công ty nhỏ và vừa)
EN_EX_list = data_4[(data_4['company_size'].isin(['S','M'])) & (data_4['experience_level'].isin(['EN','EX']))]
EN_EX_list = EN_EX_list.groupby(['job_title', 'experience_level'])['salary_in_usd'].mean().round().reset_index()

# Filter jobs that have both EN & EX experience level:
EN_EX_job_count = EN_EX_list['job_title'].value_counts()
valid_EN_EX_jobs = EN_EX_job_count[EN_EX_job_count == 2].index
final_EN_EX_list = EN_EX_list[EN_EX_list['job_title'].isin(valid_EN_EX_jobs)]

# Pivot table & find the salary gap:
result_Ex4_11 = final_EN_EX_list.pivot(index = 'job_title', columns = 'experience_level', values = 'salary_in_usd')
result_Ex4_11['salary_gap'] = abs(result_Ex4_11['EN'] - result_Ex4_11['EX'])

# Find the job with the biggest salary gap:
result_Ex4_11 = result_Ex4_11[result_Ex4_11['salary_gap'] == result_Ex4_11['salary_gap'].max()].reset_index()
print(f"Job có chênh lệch thu nhập trung bình giữa EN & EX lớn nhất trong các công ty nhỏ và vừa:")
for index, row in result_Ex4_11.iterrows():
    print(f"{row['job_title']} với gap thu nhập {row['salary_gap']} USD")

In [ ]:
# Task 12: Nước nào trả lương cao nhất cho nhân sự nước ngoài
foreign_staff_list = data_4[data_4['company_location'] != data_4['employee_residence']]
foreign_staff_pay_by_country = foreign_staff_list.groupby('company_location')['salary_in_usd'].mean().round()
result_Ex4_12 = foreign_staff_pay_by_country.idxmax()

print(f'Nước trả lương cao nhất cho nhân sự nước ngoài: {result_Ex4_12}')

In [ ]:
# Task 13: Có phải cùng experience_level, công ty quy mô càng lớn trả lương càng cao không
result_Ex4_13 = data_4.groupby('experience_level').apply(lambda x: x.groupby('company_size')['salary_in_usd'].mean().round().sort_values(ascending= False))
print(result_Ex4_13)

#Kết luận: Với cùng experience_level, công ty quy mô càng lớn có xu hướng trả lương càng cao

In [ ]:
# Task 14: Sắp xếp các job title theo thứ tự tăng dần về thu nhập trung bình
pay_by_job = data_4.groupby('job_title')['salary_in_usd'].mean().round().sort_values()
print(pay_by_job)

In [ ]:
# Task 15: Liệt kê những job title có mức thu nhập trung bình cao nhất ở 1 quốc gia, nhưng 
# lại nằm trong top 3 job có thu nhập trung bình thấp nhất ở 1 quốc gia khác
country_list = data_4['company_location'].unique()

#Iterate through each country in the country list:
for country in country_list:
    # Find the highest paying job in that country:
    top_job = data_4[data_4['company_location'] == country].groupby('job_title')['salary_in_usd'].mean().round().sort_values(ascending = False).index[0]
    other_country_list = data_4[data_4['company_location'] != country]['company_location'].unique()

    # Iterate through other country list:
    for other_country in other_country_list: 
        bottom_jobs = data_4[data_4['company_location'] == other_country].groupby('job_title')['salary_in_usd'].mean().round().sort_values(ascending = True).head(3).index.tolist()

        # Check if that job in the bottom 3 jobs in another country:
        if top_job in bottom_jobs :
            print(f'{top_job} is the highest-paying job in {country} but is among the 3 lowest-paying job in {other_country}')